# TRADING INTELLIGENT · CH06 — LABORATOIRE G1
## **Visite guidée intégrale : du premier neurone à une décision de trading**

**Objectif :** apprendre à *lire* un réseau et *anticiper* sa sortie, comprendre les formats d'entrée/sortie, observer l'apprentissage, puis expérimenter avec MLP, RNN, GRU et LSTM. Ce laboratoire est **guidé** : les exemples de référence sont déjà programmés ; votre travail est de prévoir, exécuter, modifier une variable à la fois et interpréter.

**Les 5 étapes** : ① neurone/MLP, ② séries→fenêtres/tenseurs, ③ mémoire RNN/LSTM/GRU, ④ comparaison, ⑤ prévision→trading après coûts.

**Données exclusivement fictives** ; aucune API ni clé ; **CPU** dans Colab. Les solutions analytiques réservées à MASTER sont absentes de STUDENT. Pour la construction autonome, voir **Laboratoire A2**.

**Mode opératoire :** avant chaque expérience, énoncer une prédiction ; exécuter le code ; lire les *shapes* et les axes ; noter l'écart ; ensuite seulement modifier un hyperparamètre. Les cases « À vous » de ce laboratoire sont des micro-expériences et ne bloquent pas son exécution.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, brier_score_loss
SEED = 20261123
np.random.seed(SEED);torch.manual_seed(SEED)
torch.set_num_threads(min(2,torch.get_num_threads()))
plt.rcParams.update({'figure.figsize':(8.5,3.5),'axes.grid':True,'grid.alpha':.16,'axes.spines.right':False,'axes.spines.top':False})
print('PyTorch',torch.__version__,'| mode CPU, reproductibilité des graines (non bit-à-bit sur tout matériel)')

## Parcours 1 · Pourquoi un neurone puis un MLP ?
**Question à anticiper :** que signifie chaque coefficient de `nn.Linear`, pourquoi un MLP peut apprendre une relation non linéaire, et comment la perte descend-elle ?

**Boussole des dimensions :** `X(batch, features)` → `MLP` → `y_hat(batch, sorties)`. Dès le départ, pensez en *formes de tenseurs*.

## A — Un neurone que je peux recalculer sur papier

Deux entrées déjà normalisées : momentum $x_1=0,6$, volume relatif centré $x_2=-0,2$. Paramètres fictifs $w=(0,5;-1)$, $b=0,1$. Avant d’exécuter, prévoir $z=w^\top x+b$ puis $\mathrm{ReLU}(z)$.

In [ ]:
x=np.array([.6,-.2]); w=np.array([.5,-1.]); b=.1
z=float(x@w+b); a=max(0,z)
print('Somme pondérée z =',z,'; activation ReLU =',a)
assert np.isclose(z,.6) and np.isclose(a,.6)
# Testons une entrée négative
x2=np.array([-1.,.5]);assert np.isclose(max(0,x2@w+b),0)

**À vous — Refaire le calcul pour $x=(2,-1)$, $w=(0,4,0,3)$, $b=-0,2$. Quelle est la sortie ReLU ?**

In [ ]:
# À VOUS : compléter puis tester votre réponse.
# Votre code peut rester vide pendant l’exécution automatique du notebook.

## B — Voir les matrices réellement stockées par PyTorch

In [ ]:
dense=nn.Linear(2,1)
with torch.no_grad():
    dense.weight.copy_(torch.tensor([[.5,-1.]]))
    dense.bias.fill_(.1)
X=torch.tensor([[.6,-.2],[-1.,.5]],dtype=torch.float32)
Z=dense(X)
print('X:',tuple(X.shape),'poids:',tuple(dense.weight.shape),'biais:',tuple(dense.bias.shape),'sorties:',tuple(Z.shape))
print(Z.detach().numpy().ravel())
assert Z.shape==(2,1) and torch.allclose(Z[:,0],torch.tensor([.6,-.9]))

### MLP : 12 entrées, 8 neurones, 1 sortie

In [ ]:
mlp_demo=nn.Sequential(nn.Linear(12,8),nn.ReLU(),nn.Linear(8,1))
X_demo=torch.zeros(32,12); out=mlp_demo(X_demo)
params=sum(p.numel() for p in mlp_demo.parameters())
print('Entrée',tuple(X_demo.shape),'Sortie',tuple(out.shape),'Paramètres',params)
assert (out.shape,params)==(torch.Size([32,1]),113)

**À vous — Pour 10 séances, 5 features et 3 rendements futurs, proposer les shapes et le nombre de paramètres d’un MLP $50\rightarrow16\rightarrow3$.**

In [ ]:
# À VOUS : compléter puis tester votre réponse.
# Votre code peut rester vide pendant l’exécution automatique du notebook.

## C — Observer réellement une descente de gradient

In [ ]:
w=torch.tensor([1.],requires_grad=True)
eta=.1; journal=[]
for k in range(10):
    pred=w*2.
    loss=.5*(pred-4.)**2
    journal.append((k,float(w.detach()),float(loss.detach())))
    loss.backward()
    with torch.no_grad():w -= eta*w.grad
    w.grad.zero_()
print(pd.DataFrame(journal,columns=['Étape','Poids','Perte']).head().to_string(index=False))
assert journal[1][1] > journal[0][1]
plt.plot([z[0] for z in journal],[z[2] for z in journal],'-o');plt.xlabel('Mise à jour');plt.ylabel('Perte');plt.title('La perte diminue dans cet exemple scalaire');plt.show()

## D — Un MLP apprend-il une relation qu’une droite ne voit pas ?

In [ ]:
rng=np.random.default_rng(SEED)
xx=rng.uniform(-2,2,400)
yy=xx**2+.18*rng.normal(size=len(xx))
# Ici, pas de temps : permutation préalable autorisée pour cet exercice mathématique.
perm=rng.permutation(len(xx));xx,yy=xx[perm],yy[perm]
a,b=240,320
xt=torch.tensor(xx.reshape(-1,1),dtype=torch.float32)
yt=torch.tensor(yy.reshape(-1,1),dtype=torch.float32)
train=(xt[:a],yt[:a]);val=(xt[a:b],yt[a:b]);test=(xt[b:],yt[b:])
plt.scatter(xx[:a],yy[:a],s=9,alpha=.45,label='TRAIN');plt.xlabel('x');plt.ylabel('x² + bruit');plt.legend();plt.show()

In [ ]:
class Reseau(nn.Module):
    def __init__(self,hidden=12):
        super().__init__();self.net=nn.Sequential(nn.Linear(1,hidden),nn.Tanh(),nn.Linear(hidden,1))
    def forward(self,x):return self.net(x)
def entrainer(model,train,val,epochs=160,lr=.02):
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    hist=[];best=np.inf;best_state=None
    for epoch in range(epochs):
        model.train();opt.zero_grad();pred=model(train[0]);loss=((pred-train[1])**2).mean();loss.backward();opt.step()
        model.eval()
        with torch.no_grad():v=((model(val[0])-val[1])**2).mean().item()
        hist.append((loss.item(),v))
        if v<best:best=v;best_state={k:t.detach().clone() for k,t in model.state_dict().items()}
    model.load_state_dict(best_state)
    return np.asarray(hist)
torch.manual_seed(SEED);net=Reseau();history=entrainer(net,train,val)
plt.plot(history[:,0],label='TRAIN');plt.plot(history[:,1],label='VALIDATION');plt.xlabel('Epoch');plt.ylabel('MSE');plt.legend();plt.show()
print('Meilleure MSE validation :',round(float(history[:,1].min()),4))

In [ ]:
from sklearn.linear_model import LinearRegression
reg_lineaire=LinearRegression().fit(xx[:a,None],yy[:a])
with torch.no_grad():neural=net(xt).numpy().ravel()
from sklearn.metrics import mean_absolute_error
score_linear=mean_absolute_error(yy[a:b],reg_lineaire.predict(xx[a:b,None]))
score_mlp=mean_absolute_error(yy[a:b],neural[a:b])
print('MAE validation : droite',round(score_linear,4),'; MLP',round(score_mlp,4))
order=np.argsort(xx);plt.scatter(xx[a:b],yy[a:b],s=9,alpha=.4,label='VAL');plt.plot(xx[order],reg_lineaire.predict(xx[:,None])[order],label='Droite');plt.plot(xx[order],neural[order],label='MLP');plt.legend();plt.show()

**À vous — Changer la largeur cachée de 12 à 4 ou 32. Est-ce que plus de neurones donne nécessairement une meilleure validation ?**

In [ ]:
# À VOUS : compléter puis tester votre réponse.
# Votre code peut rester vide pendant l’exécution automatique du notebook.

## E — Classification binaire : logits, probabilités et seuil

In [ ]:
# Problème XOR non-linéaire sur deux features (données fictives).
rng=np.random.default_rng(SEED+1)
Xc=rng.uniform(-1,1,(600,2));yc=((Xc[:,0]*Xc[:,1])>0).astype(float)
Xtr=torch.tensor(Xc[:360],dtype=torch.float32);Ytr=torch.tensor(yc[:360,None],dtype=torch.float32)
Xva=torch.tensor(Xc[360:480],dtype=torch.float32);Yva=torch.tensor(yc[360:480,None],dtype=torch.float32)
Xte=torch.tensor(Xc[480:],dtype=torch.float32);Yte=torch.tensor(yc[480:,None],dtype=torch.float32)
modele_cls=nn.Sequential(nn.Linear(2,16),nn.Tanh(),nn.Linear(16,1))
torch.manual_seed(SEED);opt=torch.optim.Adam(modele_cls.parameters(),lr=.025);crit=nn.BCEWithLogitsLoss()
for epoch in range(90):
    modele_cls.train();opt.zero_grad();logits=modele_cls(Xtr);loss=crit(logits,Ytr);loss.backward();opt.step()
modele_cls.eval()
with torch.no_grad():
    logits_val=modele_cls(Xva);probs=torch.sigmoid(logits_val).numpy().ravel()
assert np.all((probs>=0)&(probs<=1))
print('Accuracy validation @0.5:',round(accuracy_score(Yva.numpy().ravel(),probs>.5),3))
print('Brier score validation:',round(brier_score_loss(Yva.numpy().ravel(),probs),3))

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(10,3.5))
axs[0].scatter(Xc[360:480,0],Xc[360:480,1],c=yc[360:480],cmap='coolwarm',s=16)
axs[0].set(title='Classes réelles',xlabel='x1',ylabel='x2')
axs[1].scatter(Xc[360:480,0],Xc[360:480,1],c=probs,cmap='viridis',vmin=0,vmax=1,s=16)
axs[1].set(title='Probabilité estimée par le MLP',xlabel='x1',ylabel='x2')
plt.tight_layout();plt.show()

**À vous — Comparer les seuils 0,4, 0,5 et 0,6 : les probabilités changent-elles ou uniquement les décisions ?**

In [ ]:
# À VOUS : compléter puis tester votre réponse.
# Votre code peut rester vide pendant l’exécution automatique du notebook.

### Checkpoint 1 — Trois vérifications de compréhension

1. Calculez les **113 paramètres** de `12 → 8 → 1` sans PyTorch.
2. Pourquoi `nn.Linear(12,8).weight.shape == (8,12)` ? Décrivez le sens mathématique des lignes.
3. Le seuil de classification change-t-il les probabilités ou seulement les classes ?

**Mini-défi :** entraînez à nouveau le MLP avec 4 neurones cachés, puis 32. Faites l'hypothèse avant l'exécution et comparez **VALIDATION**, jamais seulement TRAIN.

## Parcours 2 · Une série temporelle devient une matrice puis un tenseur
**Transition :** dans le MLP précédent, chaque ligne était indépendante. Pour une série, nous devons construire les lignes à partir de l'histoire passée, sans jamais utiliser la cible future dans les entrées.

Exemple : 10 rendements, lookback `L=3`, horizon `h=1` → `N = T-L-h+1 = 7` fenêtres. Le passage à `(batch,L,F)` est la passerelle vers les RNN.

## A — Faire naître un tenseur à partir des 10 rendements du CH05

In [ ]:
returns=np.array([.2,-.4,.6,.1,-.3,.5,-.2,.3,-.1,.4],dtype=float)
def make_windows(arr,L,h=1):
    X,y=[],[]
    for t in range(L-1,len(arr)-h):
        X.append(arr[t-L+1:t+1]);y.append(arr[t+h])
    return np.asarray(X),np.asarray(y)
X,y=make_windows(returns,L=3,h=1)
print('X:',X.shape,'y:',y.shape)
print(pd.DataFrame(X,columns=['t-2','t-1','t']).assign(cible_t_plus_1=y).to_string(index=False))
assert X.shape==(7,3) and y.shape==(7,)
assert X[0].tolist()==[.2,-.4,.6] and np.isclose(y[0],.1)

**À vous — Si $L=4$ et $h=2$, combien de fenêtres complètes ? Donner la première et sa cible.**

In [ ]:
# À VOUS : compléter puis tester votre réponse.
# Votre code peut rester vide pendant l’exécution automatique du notebook.

## B — Deux manières de donner le même passé

In [ ]:
B,L,F=32,4,3
x=torch.randn(B,L,F)
print('Séquentiel',x.shape,'; MLP aplati',x.reshape(B,-1).shape)
assert x.shape==(32,4,3) and x.reshape(B,-1).shape==(32,12)

## C — RNN calculé sur trois dates (un seul état)

In [ ]:
x=[.2,-.4,.6];h=0.;trace=[]
for t,v in enumerate(x,1):
    h=float(np.tanh(v+.5*h));trace.append((t,v,h))
display(pd.DataFrame(trace,columns=['t','x_t','h_t']))
assert np.isclose(trace[0][2],np.tanh(.2))
print('L’état h3 résume la séquence ; il ne contient pas explicitement la liste x1,x2,x3.')

## D — Quelles formes PyTorch renvoie-t-il ?

In [ ]:
B,L,F,H=16,20,5,8
X=torch.randn(B,L,F)
modeles={'RNN':nn.RNN(F,H,batch_first=True),'GRU':nn.GRU(F,H,batch_first=True),'LSTM':nn.LSTM(F,H,batch_first=True)}
for nom,mod in modeles.items():
    sortie,etat=mod(X)
    parametres=sum(p.numel() for p in mod.parameters())
    print(nom,'output:',tuple(sortie.shape),'etat:',tuple(etat[0].shape) if nom=='LSTM' else tuple(etat.shape),'params:',parametres)
    assert sortie.shape==(B,L,H)
assert sum(p.numel() for p in nn.RNN(3,8,batch_first=True).parameters())==104
assert sum(p.numel() for p in nn.GRU(3,8,batch_first=True).parameters())==312
assert sum(p.numel() for p in nn.LSTM(3,8,batch_first=True).parameters())==416

**À vous — Pour $B=32$, $L=15$, $F=3$, $H=10$, dessiner l’entrée, les états et la tête de trois rendements.**

In [ ]:
# À VOUS : compléter puis tester votre réponse.
# Votre code peut rester vide pendant l’exécution automatique du notebook.

### Checkpoint 2 — Distinguez les quatre dimensions

- Série univariée `L=4` : une fenêtre contient `(4,1)` ; 32 fenêtres donnent `(32,4,1)`.
- Trois caractéristiques temporelles `L=20` : entrée récurrente `(32,20,3)` ; entrée MLP équivalente `(32,60)`.
- Pour **trois rendements futurs prédits directement**, la tête de sortie a 3 unités ; la sortie finale est `(32,3)`.

**Interdit :** introduire `r[t+1]` dans une caractéristique censée être disponible après la clôture `t`.

## Parcours 3 · De la mémoire simple aux portes LSTM et GRU
**Question :** pourquoi un réseau récurrent reçoit-il un état précédent ? Que filtrent les portes ? Les équations indispensables sont un *mode d'emploi*, non un exercice de dérivation longue.

- RNN : `h_t = tanh(W_x x_t + W_h h_{t-1} + b)`.
- LSTM : état de cellule conservé/mis à jour, sortie filtrée par des portes.
- GRU : combinaison contrôlée d'ancien état et de candidat.

Nous observons les `shapes`, l'évolution des états et une expérience où la cible dépend du début et de la fin de la séquence.

## E — Portes LSTM et GRU : le calcul qui explique la mémoire

In [ ]:
c_old=.8;f=.9;i=.2;c_cand=-.5;o=.7
c_new=f*c_old+i*c_cand;h_new=o*np.tanh(c_new)
print('LSTM: c_t=',round(c_new,4),'h_t=',round(h_new,4))
h_old=.8;z=.75;h_cand=-.4
h_gru=z*h_old+(1-z)*h_cand
print('GRU: h_t=',round(h_gru,4))
assert np.isclose(c_new,.62) and np.isclose(h_gru,.5)

## F — Expérience de mémoire contrôlée : information à distance

In [ ]:
# La cible combine le PREMIER et le DERNIER terme, information connue dans la fenetre.
rng=np.random.default_rng(SEED+3)
Nseq,L=520,12
Xn=rng.normal(size=(Nseq,L,1)).astype('float32')
yn=(.7*Xn[:,0,0]+.3*Xn[:,-1,0]).astype('float32').reshape(-1,1)
Xa=torch.from_numpy(Xn);ya=torch.from_numpy(yn)
tr,va=360,440
train=(Xa[:tr],ya[:tr]);val=(Xa[tr:va],ya[tr:va]);test=(Xa[va:],ya[va:])
assert train[0].shape==(360,12,1)
class SeqReg(nn.Module):
    def __init__(self,cell='GRU',hidden=10):
        super().__init__()
        if cell=='MLP':self.rec=None;self.model=nn.Sequential(nn.Flatten(),nn.Linear(L,20),nn.Tanh(),nn.Linear(20,1))
        else:self.rec={'RNN':nn.RNN,'GRU':nn.GRU,'LSTM':nn.LSTM}[cell](1,hidden,batch_first=True);self.head=nn.Linear(hidden,1)
    def forward(self,x):
        if self.rec is None:return self.model(x)
        z,_=self.rec(x);return self.head(z[:,-1])
def fit_sequence(net,train,val,epochs=16,lr=.012):
    opt=torch.optim.Adam(net.parameters(),lr=lr);hist=[];best=np.inf;state=None
    for epoch in range(epochs):
        net.train();order=torch.randperm(len(train[0]))
        for idx in order.split(60):
            opt.zero_grad();pred=net(train[0][idx]);loss=((pred-train[1][idx])**2).mean();loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(),1.0);opt.step()
        net.eval()
        with torch.no_grad():
            tv=((net(train[0])-train[1])**2).mean().item()
            vv=((net(val[0])-val[1])**2).mean().item()
        hist.append((tv,vv))
        if vv<best:best=vv;state={k:v.detach().clone() for k,v in net.state_dict().items()}
    net.load_state_dict(state);return np.asarray(hist)
resultats={}
for architecture in ['MLP','RNN','GRU','LSTM']:
    torch.manual_seed(SEED)
    net=SeqReg(architecture);history=fit_sequence(net,train,val)
    resultats[architecture]={'modele':net,'histoire':history,'mse_val':float(history[:,1].min())}
    print(architecture,'meilleure MSE VAL',round(resultats[architecture]['mse_val'],4))
for nom,r in resultats.items():plt.plot(r['histoire'][:,1],label=nom)
plt.xlabel('Epoch');plt.ylabel('MSE validation');plt.title('Un résultat mesuré : pas un classement décidé à l’avance');plt.legend();plt.show()

**Interprétation.** Le MLP connaît aussi le premier élément de la fenêtre : il peut donc réussir ! Une cellule récurrente traite les positions avec des poids partagés, mais son intérêt dépend de la tâche. Tester plusieurs graines et plusieurs longueurs si l'on veut étudier la stabilité.

**À vous — Changer la longueur de séquence (par exemple $L=6$ ou $L=20$) et expliquer quels modèles possèdent toujours l’information cible.**

In [ ]:
# À VOUS : compléter puis tester votre réponse.
# Votre code peut rester vide pendant l’exécution automatique du notebook.

## G — Many-to-one, multi-horizon et coût de la mémoire

In [ ]:
x=torch.randn(32,20,5)
gr=nn.GRU(5,12,batch_first=True)
out,h=gr(x)
head_one=nn.Linear(12,1);head_three=nn.Linear(12,3);head_many=nn.Linear(12,2)
print('Un pas',tuple(head_one(out[:,-1]).shape),'Trois horizons',tuple(head_three(out[:,-1]).shape),'Chaque date',tuple(head_many(out).shape))
assert head_many(out).shape==(32,20,2)

### Checkpoint 3 — Pas de hiérarchie magique
Un LSTM n'est pas toujours meilleur qu'un MLP ou un GRU. Dans notre expérience de mémoire, le MLP reçoit **lui aussi tous les instants de la fenêtre**. Comparez la perte de validation et discutez la taille d'échantillon, le nombre de paramètres et l'optimisation.

**Micro-expérience :** remplacez la fenêtre de 12 instants par 6 dans la **définition du dataset ET du modèle** ; revérifiez les shapes avant d'entraîner.

## Parcours 4 · Synthèse financière : de l'information réellement disponible au modèle
**Horloge scientifique** : après clôture `t`, on possède les valeurs jusqu'à `t`, on prédit un **rendement ouverture→clôture `t+1`**, on entre à l'ouverture `t+1`, puis on liquide à sa clôture. Les coûts affectent le P&L même si la prédiction est juste.

**Même protocole pour tout le monde** : baselines puis MLP/RNN/GRU/LSTM ; séparation chronologique ; scaler TRAIN ; choix VALIDATION ; consultation finale de TEST ; signal, frais et limites. Ne confondez jamais score prédictif et profit garanti.

## A — Un marché synthétique avec une vraie horloge de décision

In [ ]:
# Lecture locale si possible ; sinon génération autonome exactement selon les mêmes règles.
from pathlib import Path
file_candidates=[Path('CH06_MARCHE_FICTIF_V1.csv'),Path('../datasets/CH06_MARCHE_FICTIF_V1.csv'),Path('datasets/CH06_MARCHE_FICTIF_V1.csv')]
def generate_synthetic_market():
    rng=np.random.default_rng(20261123);n=960
    dates=pd.bdate_range('2021-01-04',periods=n)
    gap=np.zeros(n);intraday=np.zeros(n);volume=np.zeros(n);op=np.zeros(n);cl=np.zeros(n)
    vdev=rng.normal(0,.33,n)
    for t in range(n):
        scale=[.42,.68,.52][(t//160)%3]
        prev=intraday[t-1] if t else 0.
        lag4=intraday[t-4] if t>=4 else 0.
        vol_effect=vdev[t-1] if t else 0.
        intraday[t]=.12*np.tanh(.9*prev-.5*lag4+.65*vol_effect)+rng.normal(0,scale)
        gap[t]=rng.normal(0,.10)
        volume[t]=int(1_000_000*np.exp(vdev[t]+.07*abs(prev)))
        op[t]=(cl[t-1] if t else 100.)*np.exp(gap[t]/100)
        cl[t]=op[t]*np.exp(intraday[t]/100)
    return pd.DataFrame({'date':dates.strftime('%Y-%m-%d'),'open':op,'close':cl,'volume':volume,'r_intraday_pct':intraday,'gap_pct':gap})
path=next((p for p in file_candidates if p.exists()),None)
market=pd.read_csv(path) if path else generate_synthetic_market()
market['date']=pd.to_datetime(market['date'])
market=market.sort_values('date').reset_index(drop=True)
assert len(market)==960 and (market['open']>0).all() and (market['close']>0).all()
assert np.max(np.abs(100*np.log(market['close']/market['open'])-market['r_intraday_pct']))<1e-7
print('Source:',str(path) if path else 'regénération autonome','; séances',len(market))
print(market[['date','open','close','volume']].head().to_string(index=False))
fig,axs=plt.subplots(1,2,figsize=(11,3.4));axs[0].plot(market.date,market.close);axs[0].set(title='Cours fictifs',ylabel='Unité de prix')
axs[1].plot(market.date,market.r_intraday_pct);axs[1].set(title='Rendements ouverture-clôture',ylabel='Points de %');plt.tight_layout();plt.show()

## B — Construire seulement des caractéristiques disponibles à la clôture t

In [ ]:
d=market.copy()
d['ret_cc']=100*np.log(d['close']/d['close'].shift(1))
d['ret_oc']=d['r_intraday_pct']
d['vol_rel']=d['volume']/d['volume'].shift(1).rolling(5).mean()
d['sigma5']=d['ret_cc'].rolling(5).std(ddof=1)
d['mom5']=100*np.log(d['close']/d['close'].shift(5))
features=['ret_cc','ret_oc','vol_rel','sigma5','mom5']
# label : rendement ouverture-cloture de la prochaine seance, NON connu apres close t
# Il sera realise et devient mature a close t+1.
d['y_future']=d['ret_oc'].shift(-1)
d['target_index']=np.arange(len(d))+1
usable=d.dropna(subset=features+['y_future']).copy()
print('Première décision',usable.date.iloc[0].date(),'dernière décision',usable.date.iloc[-1].date())
display(usable[['date']+features+['y_future']].head(3).round(4))
assert usable['target_index'].max()==len(market)-1
assert np.all(np.isfinite(usable[features].to_numpy()))

**À vous — Expliquer pourquoi un ratio de volume utilise la moyenne des cinq séances PRÉCÉDENTES et non une moyenne centrée.**

In [ ]:
# À VOUS : compléter puis tester votre réponse.
# Votre code peut rester vide pendant l’exécution automatique du notebook.

## C — Fenêtres multivariées et séparation par dates de réalisation

In [ ]:
L=20;F=len(features)
base=usable.reset_index(drop=True)
raw=base[features].to_numpy(dtype=np.float32)
Xw=[];yw=[];decision_idx=[];label_idx=[];dates=[]
for end in range(L-1,len(base)):
    Xw.append(raw[end-L+1:end+1]);yw.append(float(base.y_future.iloc[end]))
    decision_idx.append(int(base.target_index.iloc[end])-1)
    label_idx.append(int(base.target_index.iloc[end]))
    dates.append(base.date.iloc[end])
Xw=np.asarray(Xw,dtype='float32');yw=np.asarray(yw,dtype='float32').reshape(-1,1)
decision_idx=np.asarray(decision_idx);label_idx=np.asarray(label_idx)
N=len(Xw);s1=int(.60*N);s2=int(.80*N)
first_val_date_index=int(decision_idx[s1]);first_test_date_index=int(decision_idx[s2])
# modèle gelé AVANT la première décision du bloc suivant : toutes les cibles sont matures
itr=np.where((np.arange(N)<s1)&(label_idx<first_val_date_index))[0]
iva=np.where((np.arange(N)>=s1)&(np.arange(N)<s2)&(label_idx<first_test_date_index))[0]
ite=np.where(np.arange(N)>=s2)[0]
assert max(label_idx[itr])<first_val_date_index
assert max(label_idx[iva])<first_test_date_index
assert max(itr)<min(iva) and max(iva)<min(ite)
print('X',Xw.shape,'y',yw.shape)
print('TRAIN',len(itr),'VAL',len(iva),'TEST',len(ite),'deux lignes frontière exclues au besoin')
print('Première décision VALIDATION :',dates[s1],'Première décision TEST :',dates[s2])

## D — Normalisation entraînée uniquement sur TRAIN

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
# Ajustement uniquement sur les X TRAIN ; aucune information des dates VAL/TEST.
scaler.fit(Xw[itr].reshape(-1,F))
Xscaled=scaler.transform(Xw.reshape(-1,F)).reshape(Xw.shape).astype(np.float32)
Xtr,Xva,Xte=(torch.from_numpy(Xscaled[z]) for z in (itr,iva,ite))
ytr,yva,yte=(torch.from_numpy(yw[z]) for z in (itr,iva,ite))
assert Xtr.shape[1:]==(L,F)
print('MLP input',tuple(Xtr.reshape(len(Xtr),-1).shape),'| RNN input',tuple(Xtr.shape),'| cible',tuple(ytr.shape))
print('Moyennes historiques entraînement',np.round(scaler.mean_,3))

## E — Baselines : le zéro et Ridge, avant les réseaux

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
Xtr2=Xtr.numpy().reshape(len(Xtr),-1);Xva2=Xva.numpy().reshape(len(Xva),-1)
base_zero=np.zeros(len(yva))
ridge=Ridge(alpha=10.).fit(Xtr2,ytr.numpy().ravel())
pred_ridge_val=ridge.predict(Xva2)
print('MAE VALIDATION: zéro',round(mean_absolute_error(yva,base_zero),4),'| Ridge',round(mean_absolute_error(yva,pred_ridge_val),4))

## F — Même contrat d’apprentissage pour MLP, RNN, GRU et LSTM

In [ ]:
class ForecastNet(nn.Module):
    def __init__(self,kind,input_features=5,lookback=20,hidden=12):
        super().__init__();self.kind=kind
        if kind=='MLP':
            self.seq=None
            self.mlp=nn.Sequential(nn.Flatten(),nn.Linear(lookback*input_features,hidden),nn.ReLU(),nn.Linear(hidden,1))
        else:
            block={'RNN':nn.RNN,'GRU':nn.GRU,'LSTM':nn.LSTM}[kind]
            self.seq=block(input_features,hidden,batch_first=True)
            self.head=nn.Linear(hidden,1)
    def forward(self,x):
        if self.kind=='MLP':return self.mlp(x)
        out,_=self.seq(x)
        return self.head(out[:,-1,:])

def train_network(kind,epochs=10,lr=.006,patience=4):
    torch.manual_seed(SEED)
    net=ForecastNet(kind,F,L,hidden=12);opt=torch.optim.Adam(net.parameters(),lr=lr)
    history=[];best=np.inf;best_state=None;wait=0
    for epoch in range(epochs):
        net.train()
        for idx in torch.randperm(len(Xtr)).split(64):
            opt.zero_grad();out=net(Xtr[idx]);loss=((out-ytr[idx])**2).mean()
            loss.backward();nn.utils.clip_grad_norm_(net.parameters(),1.0);opt.step()
        net.eval()
        with torch.no_grad():
            loss_val=((net(Xva)-yva)**2).mean().item()
            loss_tr=((net(Xtr)-ytr)**2).mean().item()
        history.append((loss_tr,loss_val))
        if loss_val < best-1e-8:
            best=loss_val;best_state={k:v.detach().clone() for k,v in net.state_dict().items()};wait=0
        else:wait+=1
        if wait>=patience:break
    net.load_state_dict(best_state)
    return net,np.asarray(history)
trained={}
for name in ['MLP','RNN','GRU','LSTM']:
    net,hist=train_network(name)
    with torch.no_grad():v=net(Xva).numpy().ravel()
    trained[name]={'net':net,'hist':hist,'val_pred':v}
    print(name,'params',sum(p.numel() for p in net.parameters()),'epochs',len(hist),'MAE VAL',round(mean_absolute_error(yva,v),4))
for name,r in trained.items():plt.plot(r['hist'][:,1],label=name)
plt.legend();plt.xlabel('Epoch');plt.ylabel('MSE validation');plt.title('Mesures réelles, mêmes dates et même cible');plt.show()

## G — Choisir un seul modèle sur validation, AVANT de consulter TEST

In [ ]:
pred_val={'Zéro':base_zero,'Ridge':pred_ridge_val}
pred_val.update({k:r['val_pred'] for k,r in trained.items()})
val_results=pd.DataFrame([{'Modèle':k,'MAE VAL':mean_absolute_error(yva,v),'RMSE VAL':np.sqrt(mean_squared_error(yva,v))} for k,v in pred_val.items()]).sort_values('MAE VAL')
display(val_results.round(4))
chosen=str(val_results.iloc[0]['Modèle'])
print('Choix gelé sur validation :',chosen)
assert chosen in pred_val

**Point de contrôle :** le code n'a pas encore regardé la cible TEST pour choisir `chosen`. Il est impossible de sélectionner le meilleur modèle sur TEST sans biaiser l'interprétation du score final.

## H — Consultation unique du test final, après sélection

In [ ]:
# On conserve le modèle choisi sur validation ; aucune re-optimisation fondée sur TEST.
if chosen=='Zéro':pred_test=np.zeros(len(yte))
elif chosen=='Ridge':pred_test=ridge.predict(Xte.numpy().reshape(len(Xte),-1))
else:
    trained[chosen]['net'].eval()
    with torch.no_grad():pred_test=trained[chosen]['net'](Xte).numpy().ravel()
y_test=yte.numpy().ravel()
print('Modèle gelé :',chosen,'| MAE TEST',round(mean_absolute_error(y_test,pred_test),4),
      '| RMSE TEST',round(np.sqrt(mean_squared_error(y_test,pred_test)),4))
plt.scatter(y_test,pred_test,s=14,alpha=.55);plt.axhline(0,color='gray',lw=.8);plt.axvline(0,color='gray',lw=.8)
plt.xlabel('Rendement réalisé ouverture→clôture t+1 (%)');plt.ylabel('Prédiction prise après clôture t (%)');plt.title('Le test ne sert PAS à rechoisir le modèle');plt.show()

## I — Transformer la prévision en position, puis appliquer des coûts

In [ ]:
# Seuil choisi sur VALIDATION seulement (exemple 0.10 point de %).
# L'expérience ne cherche pas le seuil optimal sur TEST.
tau=.10
cost_roundtrip_pct=.12
q=np.where(pred_test>tau,1,np.where(pred_test<-tau,-1,0))
# rendement intraday réel exprimé initialement en % logarithmique.
# converti en rendement simple avant calcul des gains de position.
ret_simple=np.expm1(y_test/100.)
net_pnl_frac=q*ret_simple-(cost_roundtrip_pct/100.)*np.abs(q)
benchmark_long=ret_simple-cost_roundtrip_pct/100.
assert len(q)==len(y_test) and np.all(np.isin(q,[-1,0,1]))
# capital investi selon une convention simplifiée : réinvestissement quotidien.
equity=np.cumprod(1+net_pnl_frac)
equity_long=np.cumprod(1+benchmark_long)
print('Transactions non neutres :',int(np.count_nonzero(q)),'sur',len(q),'jours TEST')
print('Rendement net composé de la stratégie expérimentale :',round(100*(equity[-1]-1),2),'%')
print('Achat quotidien théorique, mêmes coûts :',round(100*(equity_long[-1]-1),2),'%')
plt.plot(equity,label='Stratégie expérimentale');plt.plot(equity_long,label='Toujours acheteur');plt.axhline(1,color='gray',lw=.8)
plt.legend();plt.xlabel('Décisions TEST successives');plt.ylabel('Capital normalisé');plt.title('Simulation fictive avec coût aller-retour : aucune preuve de rentabilité réelle');plt.show()

**Pourquoi choisir cette cible ?** Le signal est calculé après clôture $t$ et la position est prise à l'ouverture $t+1$. Le P&L utilise donc le mouvement ouverture→clôture $t+1$. Si nous avions pris $\log(C_{t+1}/C_t)$ comme gain négociable, nous aurions attribué à la stratégie une variation de prix sur laquelle elle n'était pas en position.

## J — Classification financière : baseline, logistique et MLP neuronal

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,brier_score_loss
cl_tr=(ytr.numpy().ravel()>0).astype(int)
cl_va=(yva.numpy().ravel()>0).astype(int)
cl_te=(yte.numpy().ravel()>0).astype(int)
logistic=LogisticRegression(max_iter=500,C=1.,random_state=SEED)
logistic.fit(Xtr2,cl_tr)
prob_val_log=logistic.predict_proba(Xva2)[:,1]
base_prob=np.repeat(cl_tr.mean(),len(cl_va))
# Réseau neuronal avec logit brut, pas de sigmoïde dans la dernière couche.
torch.manual_seed(SEED)
clsnet=nn.Sequential(nn.Flatten(),nn.Linear(L*F,12),nn.ReLU(),nn.Linear(12,1))
optimizer=torch.optim.Adam(clsnet.parameters(),lr=.004)
loss_bce=nn.BCEWithLogitsLoss()
labels_tr=torch.tensor(cl_tr[:,None],dtype=torch.float32)
labels_va=torch.tensor(cl_va[:,None],dtype=torch.float32)
best=np.inf;state=None
for epoch in range(15):
    clsnet.train()
    for idx in torch.randperm(len(Xtr)).split(64):
        optimizer.zero_grad()
        logits=clsnet(Xtr[idx]);loss=loss_bce(logits,labels_tr[idx]);loss.backward();optimizer.step()
    clsnet.eval()
    with torch.no_grad():lv=loss_bce(clsnet(Xva),labels_va).item()
    if lv<best:best=lv;state={k:v.detach().clone() for k,v in clsnet.state_dict().items()}
clsnet.load_state_dict(state);clsnet.eval()
with torch.no_grad():prob_val_mlp=torch.sigmoid(clsnet(Xva)).numpy().ravel()
val_probs={'Fréquence historique':base_prob,'Logistique':prob_val_log,'MLP probabiliste':prob_val_mlp}
classes_val=pd.DataFrame([{'Modèle':name,'Brier VAL':brier_score_loss(cl_va,p),'Accuracy VAL':accuracy_score(cl_va,p>=.5)} for name,p in val_probs.items()]).sort_values('Brier VAL')
display(classes_val.round(4))
selected_classifier=str(classes_val.iloc[0]['Modèle'])
print('Classifieur choisi sur VALIDATION:',selected_classifier)
# Consultation du test uniquement après le choix.
if selected_classifier=='Logistique':
    prob_test=logistic.predict_proba(Xte.numpy().reshape(len(Xte),-1))[:,1]
elif selected_classifier=='MLP probabiliste':
    with torch.no_grad():prob_test=torch.sigmoid(clsnet(Xte)).numpy().ravel()
else:prob_test=np.repeat(cl_tr.mean(),len(cl_te))
print('Brier TEST',round(brier_score_loss(cl_te,prob_test),4),
      'Accuracy TEST',round(accuracy_score(cl_te,prob_test>=.5),4))

**À vous — Refaire le calcul d’un coût aller-retour de 0,25 % au lieu de 0,12 %, sans modifier le modèle. Qu’en concluez-vous ?**

In [ ]:
# À VOUS : compléter puis tester votre réponse.
# Votre code peut rester vide pendant l’exécution automatique du notebook.

## Synthèse active de la visite guidée — À présenter oralement

| Ce que vous devez savoir faire | Indice de maîtrise |
|---|---|
| Expliquer un `Linear` et les dimensions | Dessiner `12→8→1`, compter 113 paramètres |
| Construire une fenêtre de série | Retrouver `N=T-L-h+1`, le premier `X` et sa cible |
| Choisir `MLP` vs `RNN/LSTM/GRU` | Montrer `(B,L×F)` vs `(B,L,F)` |
| Choisir la tête de sortie | Rendement : linéaire ; classe : logit/BCE |
| Entraîner et comparer | Analyser TRAIN/VALIDATION et les baselines |
| Appliquer en trading | Chronologie, coûts, capital et limites |

**Le prochain laboratoire est différent** : plus aucun modèle prémonté. Vous aurez des fonctions à écrire, des assertions à réussir et un modèle à choisir **vous-mêmes**.

**Livraison du parcours guidé** : tableau synthétique d'une page + 2 figures interprétées + 3 questions critiques. Les expériences supplémentaires n'affectent pas le résultat de référence.